# Практическая работа № 7. Случайный лес

**Дисциплина:** Машинное обучение  
**Задача:** прогнозирование оттока абонентов мобильного оператора.

**Цель работы:** обучить и настроить модель Random Forest, сравнить её с одиночным деревом решений, исследовать OOB-оценку и определить признаки, удаление которых не приводит к существенному ухудшению качества модели.

**Источник данных:** `o_churn_synthetic.csv`.

**Примечание:** датасет синтетический, содержит 3000 искусственно сгенерированных абонентов и не является реальными данными оператора О! или другой телекоммуникационной компании.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    classification_report
)

from sklearn.base import clone

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("Библиотеки загружены успешно.")

Библиотеки загружены успешно.


In [ ]:
from google.colab import files
import io

uploaded = files.upload()

filename = "o_churn_synthetic.csv"

if filename not in uploaded:
    raise FileNotFoundError(
        f"Не найден файл {filename}. "
        "Проверь название загруженного CSV."
    )

df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Размер датасета:", df.shape)
display(df.head())

In [ ]:
print("Размер данных:", df.shape)

print("\nНазвания столбцов:")
print(df.columns.tolist())

print("\nТипы данных:")
print(df.dtypes)

print("\nПропущенные значения:")
print(df.isnull().sum())

print("\nРаспределение целевой переменной:")
print(df["churn"].value_counts())

print("\nДоли классов:")
print(df["churn"].value_counts(normalize=True).round(3))

print("\nКоличество дубликатов:", df.duplicated().sum())

In [ ]:
required_columns = [
    "subscriber_code", "region", "tariff", "tenure_months",
    "monthly_charge_som", "internet_gb", "support_calls_3m",
    "autopay", "days_since_topup", "topups_3m",
    "dropped_calls_pct", "balance_som", "churn"
]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(f"В датасете отсутствуют столбцы: {missing_columns}")

if df["churn"].isnull().any():
    raise ValueError("В целевой переменной churn есть пропуски.")

if not set(df["churn"].unique()).issubset({0, 1}):
    raise ValueError("churn должен содержать только значения 0 и 1.")

# Код абонента не используем как признак
X = df.drop(columns=["subscriber_code", "churn"])
y = df["churn"].astype(int)

categorical_features = ["region", "tariff"]
numeric_features = [
    col for col in X.columns
    if col not in categorical_features
]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)
print("\nДоля ушедших в train:", round(y_train.mean(), 3))
print("Доля ушедших в test:", round(y_test.mean(), 3))

In [ ]:
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer([
    ("cat", categorical_transformer, categorical_features),
    ("num", numeric_transformer, numeric_features)
])

tree_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", DecisionTreeClassifier(random_state=42))
])

tree_model.fit(X_train, y_train)

tree_proba = tree_model.predict_proba(X_test)[:, 1]
tree_pred = tree_model.predict(X_test)

tree_auc = roc_auc_score(y_test, tree_proba)
tree_f1 = f1_score(y_test, tree_pred)

print("Одиночное дерево решений")
print("ROC-AUC:", round(tree_auc, 4))
print("F1:", round(tree_f1, 4))

In [ ]:
default_rf = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
])

default_rf.fit(X_train, y_train)

default_proba = default_rf.predict_proba(X_test)[:, 1]
default_pred = default_rf.predict(X_test)

default_auc = roc_auc_score(y_test, default_proba)
default_f1 = f1_score(y_test, default_pred)

print("Случайный лес по умолчанию")
print("ROC-AUC:", round(default_auc, 4))
print("F1:", round(default_f1, 4))

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [None, 6, 10],
    "model__min_samples_leaf": [1, 5, 20],
    "model__max_features": ["sqrt", 0.5]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_

print("Лучшие параметры:")
print(grid_search.best_params_)

print("\nЛучший средний ROC-AUC на CV:")
print(round(grid_search.best_score_, 4))

In [ ]:
print("Изменение ROC-AUC после настройки:",
      round(best_auc - default_auc, 4))

print("Изменение F1 после настройки:",
      round(best_f1 - default_f1, 4))

In [ ]:
fitted_preprocessor = best_rf.named_steps["preprocessor"]
forest = best_rf.named_steps["model"]

encoded_names = fitted_preprocessor.get_feature_names_out()
encoded_importances = forest.feature_importances_

importance_map = {}

for name, importance in zip(encoded_names, encoded_importances):
    original_feature = name.split("__", 1)[1]

    if original_feature.startswith("region_"):
        original_feature = "region"
    elif original_feature.startswith("tariff_"):
        original_feature = "tariff"

    importance_map[original_feature] = (
        importance_map.get(original_feature, 0) + importance
    )

importance_df = pd.DataFrame({
    "Признак": list(importance_map.keys()),
    "Важность": list(importance_map.values())
}).sort_values("Важность", ascending=False)

display(importance_df.round(4))

plt.figure(figsize=(10, 6))
plt.barh(
    importance_df["Признак"][::-1],
    importance_df["Важность"][::-1]
)
plt.xlabel("Важность признака (сумма impurity-based importance)")
plt.ylabel("Исходный признак")
plt.title("Важность признаков случайного леса")
plt.tight_layout()
plt.show()

In [ ]:
tree_counts = [10, 25, 50, 100, 200, 400]
oob_scores = []

for n_trees in tree_counts:
    oob_model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", RandomForestClassifier(
            n_estimators=n_trees,
            oob_score=True,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ))
    ])

    oob_model.fit(X_train, y_train)

    score = oob_model.named_steps["model"].oob_score_
    oob_scores.append(score)

    print(f"Деревьев: {n_trees:3d} | OOB accuracy: {score:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(tree_counts, oob_scores, marker="o")
plt.xlabel("Количество деревьев")
plt.ylabel("OOB accuracy")
plt.title("Зависимость OOB-оценки от количества деревьев")
plt.xticks(tree_counts)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
feature_order = importance_df["Признак"].tolist()

# Сначала оцениваем исходный набор признаков
current_features = X.columns.tolist()
removal_results = []

baseline_scores = cross_val_score(
    clone(best_rf),
    X_train[current_features],
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

baseline_mean = baseline_scores.mean()
baseline_std = baseline_scores.std()

removal_results.append({
    "Удалённый признак": "Нет (все признаки)",
    "Осталось признаков": len(current_features),
    "ROC-AUC среднее": baseline_mean,
    "ROC-AUC std": baseline_std,
    "Изменение ROC-AUC": 0.0,
    "В пределах исходного разброса": True
})

# Удаляем признаки по одному, от наименее важного к наиболее важному
for feature in reversed(feature_order):
    if feature not in current_features:
        continue

    current_features.remove(feature)

    scores = cross_val_score(
        clone(best_rf),
        X_train[current_features],
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1
    )

    mean_score = scores.mean()
    std_score = scores.std()
    difference = mean_score - baseline_mean

    removal_results.append({
        "Удалённый признак": feature,
        "Осталось признаков": len(current_features),
        "ROC-AUC среднее": mean_score,
        "ROC-AUC std": std_score,
        "Изменение ROC-AUC": difference,
        "В пределах исходного разброса":
            abs(difference) <= baseline_std
    })

removal_df = pd.DataFrame(removal_results)

display(removal_df.round(4))

In [ ]:
plt.figure(figsize=(11, 6))

plt.errorbar(
    removal_df["Осталось признаков"],
    removal_df["ROC-AUC среднее"],
    yerr=removal_df["ROC-AUC std"],
    marker="o",
    capsize=4
)

plt.axhline(
    baseline_mean,
    linestyle="--",
    label="ROC-AUC всех признаков"
)

plt.xlabel("Количество оставшихся признаков")
plt.ylabel("Средний ROC-AUC на CV")
plt.title("Влияние удаления признаков на качество модели")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("ИТОГИ ПРАКТИЧЕСКОЙ РАБОТЫ № 7")
print("=" * 60)

print("\n1. СРАВНЕНИЕ МОДЕЛЕЙ")
display(comparison.round(4))

print("\n2. ЛУЧШИЕ ПАРАМЕТРЫ СЛУЧАЙНОГО ЛЕСА")
print(grid_search.best_params_)
print("Лучший CV ROC-AUC:", round(grid_search.best_score_, 4))

print("\n3. OOB-ОЦЕНКА")
for n, score in zip(tree_counts, oob_scores):
    print(f"{n:3d} деревьев: accuracy = {score:.4f}")

print("\n4. НАИБОЛЕЕ ВАЖНЫЕ ПРИЗНАКИ")
display(importance_df.head(5).round(4))

print("\n5. НАИМЕНЕЕ ВАЖНЫЕ ПРИЗНАКИ")
display(importance_df.tail(5).round(4))

print("\n6. РЕЗУЛЬТАТЫ УДАЛЕНИЯ ПРИЗНАКОВ")
display(removal_df.round(4))

print("\nНоутбук завершил анализ.")

## Вывод

В ходе практической работы была разработана модель прогнозирования оттока абонентов на синтетическом датасете, содержащем 3000 записей. Категориальные признаки были закодированы с помощью One-Hot Encoding, а обработка данных и обучение моделей организованы через Pipeline.

Было проведено сравнение одиночного дерева решений, случайного леса с параметрами по умолчанию и случайного леса с настроенными гиперпараметрами. Настройка выполнялась с помощью GridSearchCV и пятифолдовой стратифицированной кросс-валидации с оптимизацией ROC-AUC. По результатам сравнения необходимо указать, какая модель показала лучший результат на тестовой выборке по ROC-AUC и F1, а также объяснить возможные различия между метриками.

Для исследования влияния количества деревьев была построена OOB-кривая. Выбранное число деревьев обосновывается стабилизацией оценки и результатами подбора гиперпараметров.

Затем была проведена последовательная проверка удаления наименее важных признаков. При оценке учитывались средний ROC-AUC и стандартное отклонение между фолдами. Признаки, удаление которых не привело к изменению качества, превышающему исходный разброс CV-оценок, можно рассматривать как кандидатов на исключение. Однако это не доказывает их полной бесполезности для других моделей или будущих данных.

Полученные результаты позволяют определить, какие характеристики абонентов связаны с прогнозом оттока и какие данные потенциально можно не собирать. Важность признаков отражает их вклад в работу модели, но не устанавливает причинно-следственную связь с уходом абонентов.